# 🎬 MoneyPrinterTurbo — Google Colab (GPU T4 Gratuita)

**Redundância Nível 3** — usa GPU para geração mais rápida.

### Antes de rodar:
1. Vá em `Runtime → Change runtime type → GPU (T4)`
2. Configure os Secrets no ícone 🔑 (cadeado) na barra lateral:
   - `GEMINI_API_KEY` = sua chave do Google AI Studio
   - `PEXELS_API_KEY` = sua chave do Pexels (opcional)
3. Execute todas as células em ordem

In [ ]:
# Célula 1: Instalar FFmpeg e dependências do sistema
!apt-get update -qq
!apt-get install -y -qq ffmpeg
print('✅ FFmpeg instalado')

In [ ]:
# Célula 2: Clonar o repositório
import os
if not os.path.exists('/content/MoneyPrinterTurbo'):
    !git clone https://github.com/harry0703/MoneyPrinterTurbo /content/MoneyPrinterTurbo
%cd /content/MoneyPrinterTurbo
print('✅ Repositório clonado')

In [ ]:
# Célula 3: Instalar dependências (sem Whisper pesado)
!pip install -q moviepy==2.2.1 streamlit==1.59.1 edge_tts==7.2.7 \
    fastapi==0.136.3 uvicorn==0.32.1 openai==2.24.0 \
    loguru==0.7.3 google-genai==2.11.0 redis==5.2.0 \
    python-multipart==0.0.27 pyyaml==6.0.3 requests==2.33.1 \
    packaging==24.2 pydub==0.25.1 litellm==1.86.2 streamlit-tour==1.1.0
print('✅ Dependências instaladas')

In [ ]:
# Célula 4: Configurar chaves via Colab Secrets
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
    PEXELS_KEY = userdata.get('PEXELS_API_KEY') or ''
    print(f'✅ Gemini key: {GEMINI_KEY[:8]}...')
except Exception:
    # Fallback: definir manualmente
    GEMINI_KEY = 'AQ.Ab8RN6I9nBD55JdI9r0fOHyjkEfQ6Ek92x0TwCye0UVpV3nVZw'
    PEXELS_KEY = ''
    print('⚠️  Usando chave Gemini padrão (configurar Secrets para produção)')

In [ ]:
# Célula 5: Criar config.toml
config_content = f'''log_level = "INFO"
listen_host = "0.0.0.0"
listen_port = 8080

[app]
api_key = ""
video_source = "pexels"
pexels_api_keys = ["{PEXELS_KEY}"]
pixabay_api_keys = []
llm_provider = "gemini"
gemini_api_key = "{GEMINI_KEY}"
gemini_model_name = "gemini-2.5-flash"
subtitle_provider = "edge"
material_directory = "/tmp/mpt_materials"
enable_redis = false
max_concurrent_tasks = 2

[whisper]
model_size = "base"
device = "cuda"
compute_type = "float16"

[proxy]
[azure]
speech_key = ""
speech_region = ""
[siliconflow]
api_key = ""
[minimax_tts]
api_key = ""
base_url = ""
[elevenlabs]
api_key = ""
[ui]
hide_log = false
open_task_folder_on_completion = false
'''

with open('config.toml', 'w') as f:
    f.write(config_content)
print('✅ config.toml criado com Gemini + Edge TTS')

In [ ]:
# Célula 6: Expor via ngrok (túnel público)
!pip install -q pyngrok
from pyngrok import ngrok
import subprocess, threading

# Inicia a API em background
proc = subprocess.Popen(['python', 'main.py'])

import time; time.sleep(5)

# Cria túnel público
tunnel = ngrok.connect(8080)
print(f'🌐 URL pública da API: {tunnel.public_url}')
print(f'📖 Swagger docs: {tunnel.public_url}/docs')
print('\n💡 Compartilhe esta URL com o Cérebro IA para usar como endpoint remoto!')

In [ ]:
# Célula 7 (opcional): WebUI Streamlit também com túnel
import subprocess
stproc = subprocess.Popen([
    'streamlit', 'run', 'webui/Main.py',
    '--server.port', '8501',
    '--server.headless', 'true'
])
time.sleep(5)
ui_tunnel = ngrok.connect(8501)
print(f'🎨 WebUI Streamlit: {ui_tunnel.public_url}')